In [12]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import os
import re
import pandas as pd
import pingouin as pg
from utils_statistical_tests import plot_equivalence_heatmap

%matplotlib qt


# =============================================================================
# SETUP
# =============================================================================

metrics_time = ["HRV_MeanNN", "HRV_SDNN"]
titles_time = ["Mean NN", "SDNN"]

metrics_freq = ["HRV_LFn", "HRV_HFn", "HRV_LFHF", "HRV_VLF", "HRV_LF", "HRV_HF"]
titles_freq = ["LFn", "HFn", "LF/HF", "VLF", "LF", "HF"]

metrics_nonlinear = ["HRV_ApEn", "HRV_SampEn", "HRV_DFA_alpha1", "HRV_DFA_alpha2", "HRV_SD1", "HRV_SD2", "HRV_SD1SD2"]
titles_nonlinear = ["ApEn", "SampEn", "DFA α₁", "DFA α₂", "SD1", "SD2", "SD1/SD2"]

all_metrics = metrics_time + metrics_freq + metrics_nonlinear
all_titles = titles_time + titles_freq + titles_nonlinear
metric_to_title = dict(zip(all_metrics, all_titles))


# Define the equivalence margin (epsilon) for the TOST test
eps = 0.05
eps = 0.2


# Define the base directory to data
base_path = os.path.join("..", "Data")

# Define paths for the two experimental conditions
methods = {
    "All Windows": os.path.join(base_path, "All Windows"),
    "Method 2": os.path.join(base_path, "Window Removal")
}

# List subjects based on HRV folder of 'All Windows'
subjects_dir = os.path.join(methods["All Windows"], "HRV")
subjects = [f for f in os.listdir(subjects_dir) if f.endswith(".csv")]
subjects = sorted(subjects, key=lambda x: int(re.findall(r'\d+', x)[0]))  # sort numerically


In [ ]:
### MEDIAN AND IQR BETWEEN HRV AND PRV FOR EACH SUBJECT ###

results = {m: {} for m in methods}     # True/False equivalence
p_values = {m: {} for m in methods}    # p-values

# ---- Loop on subjects ----
for subject_file in subjects:
    subj_name = subject_file.replace(".csv", "")

    # ---- Loop on methods ----
    for method_name, method_path in methods.items():

        # Load HRV & PRV 
        hrv_path = os.path.join(method_path, "HRV", subject_file)
        prv_path = os.path.join(method_path, "PRV", subject_file)
        
        df_hrv = pd.read_csv(hrv_path).drop(columns='Window', errors='ignore')
        df_prv = pd.read_csv(prv_path).drop(columns='Window', errors='ignore')

        results_per_subj = []

        # ---- Loop on metrics ----
        for metric in all_metrics:

            hrv = df_hrv[metric].dropna()
            prv = df_prv[metric].dropna()

            # Combine HRV + PRV and compute median/IQR
            combined = pd.concat([hrv, prv], ignore_index=True).dropna()
            median_val = combined.median()
            iqr_val = combined.quantile(0.75) - combined.quantile(0.25)

            # Robust scaling
            hrv_norm = (hrv - median_val) / iqr_val
            prv_norm = (prv - median_val) / iqr_val

            # Run paired TOST 
            tost_res = pg.tost(x=hrv_norm, y=prv_norm, bound=eps, paired=True)
            pval = tost_res["pval"].values[0]

            results_per_subj.append({
                "Feature": metric,
                "p_value": pval,
                "epsilon": eps,
                "dof": tost_res["dof"].values[0],
                "Equivalent": pval < 0.05 
            })

        # Convert list to dataframe for this subject
        tost_result = pd.DataFrame(results_per_subj).set_index("Feature")
        # Save results 
        results[method_name][subj_name] = tost_result["Equivalent"]
        p_values[method_name][subj_name] = tost_result["p_value"]


# Convert results to DataFrames
df_equiv_all = pd.DataFrame(results["All Windows"])
df_equiv_method2 = pd.DataFrame(results["Method 2"])

# Convert p-values to DataFrame (features × subjects)
df_pvalues_all = pd.DataFrame(p_values["All Windows"])
df_pvalues_method2 = pd.DataFrame(p_values["Method 2"])


# =============================================================================
# VISUALIZATION
# =============================================================================

for method_name in methods:
    df_equiv = pd.DataFrame(results[method_name])
    plot_equivalence_heatmap(df_equiv, method_name=method_name, epsilon=eps)

# Summarize number of subjects showing equivalence per feature
summary_df = pd.DataFrame({
    'Feature': df_equiv_all.index,
    'All Windows': df_equiv_all.sum(axis=1),
    'Method 2': df_equiv_method2.sum(axis=1)
}).set_index('Feature')

print(f"\n=== Equivalence Summary (ε = {eps}) ===")
print(summary_df)


# =============================================================================
# SAVING IN CSV
# =============================================================================

#df_pvalues_all.to_csv(f"C:/Users/ilari/Downloads/TOST_pvalues_AllWindows_per_subj ε={eps}.csv")
#df_pvalues_method2.to_csv(f"C:/Users/ilari/Downloads/TOST_pvalues_Method2_per_subj ε={eps}.csv")




=== Equivalence Summary (ε = 0.05) ===
                All Windows  Method 2
Feature                              
HRV_MeanNN               39        49
HRV_SDNN                  4        12
HRV_LFn                   0         2
HRV_HFn                   0         2
HRV_LFHF                  0         0
HRV_VLF                   1         5
HRV_LF                    0         4
HRV_HF                    2         2
HRV_ApEn                  0         0
HRV_SampEn                2         1
HRV_DFA_alpha1            0         0
HRV_DFA_alpha2            2         9
HRV_SD1                   0         2
HRV_SD2                   6        20
HRV_SD1SD2                0         2


In [13]:
### GLOBAL MEDIAN AND IQR BETWEEN HRV AND PRV (ACROSS ALL SUBJECTS) ###

# =============================================================================
# MEDIAN AND IQR ACROSS ALL SUBJS
# =============================================================================

# dict: for each method → for each metric → list of global values
all_values = {
    method_name: {metric: [] for metric in all_metrics}
    for method_name in methods
}

### Collects all HRV and PRV values globally for each metric and method
for subject_file in subjects:
    for method_name, method_path in methods.items():

        hrv_path = os.path.join(method_path, "HRV", subject_file)
        prv_path = os.path.join(method_path, "PRV", subject_file)

        df_hrv = pd.read_csv(hrv_path).drop(columns='Window', errors='ignore')
        df_prv = pd.read_csv(prv_path).drop(columns='Window', errors='ignore')

        for metric in all_metrics:
            all_values[method_name][metric].extend(df_hrv[metric].dropna().tolist())
            all_values[method_name][metric].extend(df_prv[metric].dropna().tolist())

### Compute global median and iqr
global_medians = {method: {} for method in methods}
global_iqrs = {method: {} for method in methods}

for method_name in methods:
    for metric in all_metrics:
        series = pd.Series(all_values[method_name][metric]).dropna()
        global_medians[method_name][metric] = series.median()
        global_iqrs[method_name][metric] = series.quantile(0.75) - series.quantile(0.25)



# =============================================================================
# TOST
# =============================================================================

results = {m: {} for m in methods}     # True/False equivalence
p_values = {m: {} for m in methods}    # p-values

for subject_file in subjects:
    subj_name = subject_file.replace(".csv", "")

    for method_name, method_path in methods.items():

        hrv_path = os.path.join(method_path, "HRV", subject_file)
        prv_path = os.path.join(method_path, "PRV", subject_file)

        df_hrv = pd.read_csv(hrv_path).drop(columns='Window', errors='ignore')
        df_prv = pd.read_csv(prv_path).drop(columns='Window', errors='ignore')

        results_per_subj = []

        for metric in all_metrics:

            hrv = df_hrv[metric].dropna()
            prv = df_prv[metric].dropna()

            # Median e iqr for each metric
            median_val = global_medians[method_name][metric]
            iqr_val = global_iqrs[method_name][metric]

            # Robust scaling
            hrv_norm = (hrv - median_val) / iqr_val
            prv_norm = (prv - median_val) / iqr_val

            # Run paired TOST 
            tost_res = pg.tost(x=hrv_norm, y=prv_norm, bound=eps, paired=True)
            pval = tost_res["pval"].values[0]

            results_per_subj.append({
                "Feature": metric,
                "p_value": pval,
                "epsilon": eps,
                "dof": tost_res["dof"].values[0],
                "Equivalent": pval < 0.05 
            })

        # Convert list to dataframe for this subject
        tost_result = pd.DataFrame(results_per_subj).set_index("Feature")
        # Save results 
        results[method_name][subj_name] = tost_result["Equivalent"]
        p_values[method_name][subj_name] = tost_result["p_value"]


# Convert results to DataFrames
df_equiv_all = pd.DataFrame(results["All Windows"])
df_equiv_method2 = pd.DataFrame(results["Method 2"])

# Convert p-values to DataFrame (features × subjects)
df_pvalues_all = pd.DataFrame(p_values["All Windows"])
df_pvalues_method2 = pd.DataFrame(p_values["Method 2"])


# =============================================================================
# VISUALIZATION
# =============================================================================

for method_name in methods:
    df_equiv = pd.DataFrame(results[method_name])
    plot_equivalence_heatmap(df_equiv, method_name=method_name, epsilon=eps)

# Summarize number of subjects showing equivalence per feature
summary_df = pd.DataFrame({
    'Feature': df_equiv_all.index,
    'All Windows': df_equiv_all.sum(axis=1),
    'Method 2': df_equiv_method2.sum(axis=1)
}).set_index('Feature')

print(f"\n=== Equivalence Summary (ε = {eps}) ===")
print(summary_df)


# =============================================================================
# SAVING IN CSV
# =============================================================================

#df_pvalues_all.to_csv(f"C:/Users/ilari/Downloads/TOST_pvalues_AllWindows_all_subjs ε={eps}.csv")
#df_pvalues_method2.to_csv(f"C:/Users/ilari/Downloads/TOST_pvalues_Method2_all_subjs ε={eps}.csv")




=== Equivalence Summary (ε = 0.2) ===
                All Windows  Method 2
Feature                              
HRV_MeanNN               50        50
HRV_SDNN                 33        44
HRV_LFn                  37        40
HRV_HFn                  26        36
HRV_LFHF                 21        24
HRV_VLF                  45        46
HRV_LF                   26        41
HRV_HF                   18        29
HRV_ApEn                 25        26
HRV_SampEn               27        27
HRV_DFA_alpha1           22        28
HRV_DFA_alpha2           26        36
HRV_SD1                  17        31
HRV_SD2                  38        48
HRV_SD1SD2               18        23


In [ ]:
# NORMALIZATION PER SUBJ

import matplotlib.pyplot as plt

subject_file = 'S1.csv'
method_path = os.path.join(base_path, "All Windows")

hrv_path = os.path.join(method_path, "HRV", subject_file)
prv_path = os.path.join(method_path, "PRV", subject_file)

df_hrv = pd.read_csv(hrv_path).drop(columns='Window', errors='ignore')
df_prv = pd.read_csv(prv_path).drop(columns='Window', errors='ignore')
        
n_metrics = len(all_metrics)

ncols = 4  
nrows = (n_metrics + ncols - 1) // ncols  

fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4*nrows))
axes = axes.flatten()  

for i, (metric, title) in enumerate(zip(all_metrics, all_titles)):    
    ax = axes[i]

    hrv = df_hrv[metric].dropna()
    prv = df_prv[metric].dropna()

    combined = pd.concat([hrv, prv], ignore_index=True).dropna()
    median_val = combined.median()
    iqr_val = combined.quantile(0.75) - combined.quantile(0.25)

    hrv_norm = (hrv - median_val) / iqr_val
    prv_norm = (prv - median_val) / iqr_val

    # plot 
    ax.plot(hrv.values, label='HRV', color='darkorange')
    ax.plot(prv.values, label='PRV', color='blue')
    ax.plot(hrv_norm.values, label='HRV norm', color='darkorange', linestyle='--')
    ax.plot(prv_norm.values, label='PRV norm', color='blue', linestyle='--')

    ax.tick_params(axis='both', labelsize=16)
    ax.set_xticklabels([])
    ax.set_title(title, fontsize=18, fontweight='bold')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', fontsize=24, ncol=2)
fig.suptitle(f'{subject_file} - norm per subj', fontsize=20, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# NORMALIZATION ALL SUBJECTS

import matplotlib.pyplot as plt

subject_file = 'S1.csv'
method_path = os.path.join(base_path, "All Windows")
method_name = 'All Windows'

hrv_path = os.path.join(method_path, "HRV", subject_file)
prv_path = os.path.join(method_path, "PRV", subject_file)

df_hrv = pd.read_csv(hrv_path).drop(columns='Window', errors='ignore')
df_prv = pd.read_csv(prv_path).drop(columns='Window', errors='ignore')
        
n_metrics = len(all_metrics)

ncols = 4  
nrows = (n_metrics + ncols - 1) // ncols  

fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4*nrows))
axes = axes.flatten()  

for i, (metric, title) in enumerate(zip(all_metrics, all_titles)):    
    ax = axes[i]

    hrv = df_hrv[metric].dropna()
    prv = df_prv[metric].dropna()

    # Median e iqr for each metric
    median_val = global_medians[method_name][metric]
    iqr_val = global_iqrs[method_name][metric]

    # Robust scaling
    hrv_norm = (hrv - median_val) / iqr_val
    prv_norm = (prv - median_val) / iqr_val

    # plot 
    ax.plot(hrv.values, label='HRV', color='darkorange')
    ax.plot(prv.values, label='PRV', color='blue')
    ax.plot(hrv_norm.values, label='HRV norm', color='darkorange', linestyle='--')
    ax.plot(prv_norm.values, label='PRV norm', color='blue', linestyle='--')

    ax.tick_params(axis='both', labelsize=16)
    ax.set_xticklabels([])
    ax.set_title(title, fontsize=18, fontweight='bold')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', fontsize=24, ncol=2)
fig.suptitle(f'{subject_file} - norm all subjs', fontsize=20, fontweight='bold')

plt.tight_layout()
plt.show()

    